In [1]:
# Projete e execute o script que ilustra como calcular a interseção de 3 raios sensores com obstáculos retangulares no Pygame.

import pygame
import math
import numpy as np

LARGURA, ALTURA = 900, 650
FPS = 60
COR_FUNDO = (20, 24, 30)
COR_ROBO = (0, 200, 255)
COR_OBSTACULO = (180, 50, 50)
COR_RAIO_LIVRE = (0, 255, 100)
COR_RAIO_COLISAO = (255, 200, 0)
COR_ALVO = (255, 200, 0)
COR_DIRECAO = (255, 50, 50)
COR_TRAJETORIA = (100, 200, 100)

SENSOR_RANGE = 150.0
EMERGENCY_DISTANCE = 60.0
STOP_DISTANCE = 15.0


def wrap_angle(angle):
    return (angle + math.pi) % (2 * math.pi) - math.pi


class RaycastDemoRobot:
    def __init__(self, x, y, theta=0.0):
        self.x = float(x)
        self.y = float(y)
        self.theta = float(theta)
        self.sensor_angles = [-math.pi / 4, 0.0, math.pi / 4]  # Esq, Frente, Dir
        self.sensor_range = SENSOR_RANGE
        self.sensor_readings = [self.sensor_range] * 3
        self.v = 0.0
        self.omega = 0.0
        self.history = [(x, y)]

    def cast_rays(self, obstacles):
        """Verifica a interseção dos raios com obstáculos retangulares."""
        self.sensor_readings = []
        for beta in self.sensor_angles:
            angle = self.theta + beta
            min_dist = self.sensor_range

            for step in range(5, int(self.sensor_range), 4):
                rx = self.x + step * math.cos(angle)
                ry = self.y + step * math.sin(angle)

                if rx <= 0 or rx >= LARGURA or ry <= 0 or ry >= ALTURA:
                    min_dist = float(step)
                    break

                hit = False
                for obs in obstacles:
                    if obs.collidepoint(rx, ry):
                        min_dist = float(step)
                        hit = True
                        break
                if hit:
                    break
            self.sensor_readings.append(min_dist)

    def set_velocity(self, v, omega):
        self.v = v
        self.omega = omega

    def update(self, dt):
        self.theta += self.omega * dt
        self.theta = wrap_angle(self.theta)
        self.x += self.v * math.cos(self.theta) * dt
        self.y += self.v * math.sin(self.theta) * dt

        if len(self.history) == 0 or np.hypot(self.x - self.history[-1][0], self.y - self.history[-1][1]) > 3:
            self.history.append((self.x, self.y))
            if len(self.history) > 600:
                self.history.pop(0)

    def draw(self, surface, target_pos=None):
        if len(self.history) > 1:
            pygame.draw.lines(surface, COR_TRAJETORIA, False, self.history, 2)

        for i, beta in enumerate(self.sensor_angles):
            angle = self.theta + beta
            dist = self.sensor_readings[i]
            rx = self.x + dist * math.cos(angle)
            ry = self.y + dist * math.sin(angle)
            cor = COR_RAIO_COLISAO if dist < self.sensor_range else COR_RAIO_LIVRE
            pygame.draw.line(surface, cor, (int(self.x), int(self.y)), (int(rx), int(ry)), 2)
            pygame.draw.circle(surface, cor, (int(rx), int(ry)), 4)

        pos = (int(self.x), int(self.y))
        pygame.draw.circle(surface, COR_ROBO, pos, 16)
        fx = self.x + 24 * math.cos(self.theta)
        fy = self.y + 24 * math.sin(self.theta)
        pygame.draw.line(surface, COR_DIRECAO, pos, (int(fx), int(fy)), 3)

        if target_pos is not None:
            pygame.draw.circle(surface, COR_ALVO, (int(target_pos[0]), int(target_pos[1])), 6)
            pygame.draw.circle(surface, COR_ALVO, (int(target_pos[0]), int(target_pos[1])), 12, 1)


def go_to_goal(robot, target_pos):
    dx = target_pos[0] - robot.x
    dy = target_pos[1] - robot.y
    dist = math.hypot(dx, dy)

    if dist <= STOP_DISTANCE:
        return 0.0, 0.0, True

    desired_angle = math.atan2(dy, dx)
    angle_error = wrap_angle(desired_angle - robot.theta)

    kp = 2.5
    omega = kp * angle_error

    if abs(angle_error) < 0.5:
        v = 110.0
    elif abs(angle_error) < 1.0:
        v = 60.0
    else:
        v = 30.0

    return v, omega, False


def avoid_obstacle(robot, obstacles):
    robot.cast_rays(obstacles)
    distances = robot.sensor_readings
    active = any(dist < EMERGENCY_DISTANCE for dist in distances)

    if not active:
        return 0.0, 0.0, False

    left_clear = distances[0]
    right_clear = distances[2]
    front_blocked = distances[1] < EMERGENCY_DISTANCE

    if front_blocked:
        omega = 2.8 if left_clear >= right_clear else -2.8
    else:
        repulsive_torque = 0.0
        for beta, dist in zip(robot.sensor_angles, distances):
            if dist < EMERGENCY_DISTANCE:
                influence = (EMERGENCY_DISTANCE - dist) / EMERGENCY_DISTANCE
                repulsive_torque += 3.5 * influence * math.sin(beta)
        omega = repulsive_torque

    return 25.0, omega, True


def main():
    pygame.init()
    screen = pygame.display.set_mode((LARGURA, ALTURA))
    pygame.display.set_caption("Go-to-Goal com Desvio Reativo")
    clock = pygame.time.Clock()
    font = pygame.font.SysFont("monospace", 14)

    robot = RaycastDemoRobot(100, 500, 0.0)
    obstacles = [
        pygame.Rect(300, 200, 120, 260),
        pygame.Rect(570, 120, 160, 110),
        pygame.Rect(610, 420, 160, 140),
    ]

    target_pos = None
    reached_target = False

    running = True
    while running:
        dt = clock.tick(FPS) / 1000.0

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False
            if event.type == pygame.MOUSEBUTTONDOWN:
                target_pos = event.pos
                reached_target = False

        if target_pos is not None:
            dist_to_target = math.hypot(target_pos[0] - robot.x, target_pos[1] - robot.y)

            if dist_to_target <= STOP_DISTANCE:
                target_pos = None
                reached_target = True
                v_cmd = 0.0
                omega_cmd = 0.0
            else:
                robot.cast_rays(obstacles)
                if any(dist < EMERGENCY_DISTANCE for dist in robot.sensor_readings):
                    v_cmd, omega_cmd, _ = avoid_obstacle(robot, obstacles)
                else:
                    v_cmd, omega_cmd, _ = go_to_goal(robot, target_pos)
        else:
            v_cmd = 0.0
            omega_cmd = 0.0

        screen.fill(COR_FUNDO)
        for obs in obstacles:
            pygame.draw.rect(screen, COR_OBSTACULO, obs)
            pygame.draw.rect(screen, (255, 100, 100), obs, 2)

        robot.set_velocity(v_cmd, omega_cmd)
        robot.update(dt)
        robot.cast_rays(obstacles)
        robot.draw(screen, target_pos=target_pos)

        info = [
            f"Pose: x={robot.x:.1f} y={robot.y:.1f} theta={math.degrees(robot.theta):.1f} deg",
            f"v={robot.v:.1f} px/s | omega={robot.omega:.2f} rad/s",
            f"Distância ao alvo: {0 if target_pos is None else math.hypot(target_pos[0] - robot.x, target_pos[1] - robot.y):.1f} px",
            "Clique para definir um alvo.",
        ]

        if reached_target:
            info.append("Meta alcançada: distância menor que 15 px")

        for i, txt in enumerate(info):
            screen.blit(font.render(txt, True, (220, 220, 220)), (15, 15 + i * 20))

        pygame.display.flip()

    pygame.quit()


if __name__ == "__main__":
    main()
#fim do código

pygame 2.6.1 (SDL 2.28.4, Python 3.12.3)
Hello from the pygame community. https://www.pygame.org/contribute.html
